# Automatic Object Removal — keep the main person, erase the rest
**Computer Vision — Sapienza Università di Roma — A.A. 2025–2026**

Omri Ben Zion & Johannes Erhard · improved Kaggle pipeline

**Goal:** fully automatic *object removal* on our own Rome photos. Detect every person, keep the
main subject, and **diffuse the background into the others' place** (not a new person).

**Pipeline:** Mask R-CNN (find all people) → pick main subject → SAM (refine removal masks) →
BLIP caption (background) → removal-tuned SD-1.5 inpainting → composite back at full resolution.

**Code structure (mandatory order):** §1 Imports → §2 Globals → §3 Utils → §4 Data → §5 Network → §6 Pipeline → §7 Evaluation

## § 1 · Imports

In [ ]:
# Kaggle install — refresh the diffusion stack WITHOUT touching torch/torchvision.
# (An unconstrained -U pulls a PyPI torch build that mismatches the Kaggle GPU
#  -> "CUDA: no kernel image available". So we pin torch/torchvision to Kaggle's.)
import sys, subprocess, torch, torchvision
with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch.__version__.split('+')[0]}\n")
    f.write(f"torchvision=={torchvision.__version__.split('+')[0]}\n")
print("pinning", torch.__version__, torchvision.__version__)
def _pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *a],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "|", r.stderr.strip()[-200:])
_pip("-q", "-U", "diffusers>=0.31.0", "transformers>=4.45.0",
     "accelerate>=0.34.0", "safetensors", "-c", "/tmp/constraints.txt")
print("pip done")

In [ ]:
import os, json, gc, math, time, random, zipfile
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

from PIL import Image, ImageDraw, ImageFont, ImageFilter
import matplotlib.pyplot as plt

from transformers import SamModel, SamProcessor, BlipProcessor, BlipForConditionalGeneration
from diffusers import StableDiffusionInpaintPipeline

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## § 2 · Globals

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
# P100 has half-rate fp16 -> fp16 inference is pathologically slow there; use fp32.
DTYPE = torch.float32 if ("P100" in GPU_NAME or DEVICE == "cpu") else torch.float16
print("Device:", DEVICE, "| GPU:", GPU_NAME, "| dtype:", DTYPE)

# ── paths ───────────────────────────────────────────────────────────────────
INPUT_DIR = Path("/kaggle/input/cv-rome-removal-photos")
OUT_DIR   = Path("/kaggle/working/steps"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── diffusion working resolution (3:2, multiples of 8) ──────────────────────
PROC_W, PROC_H   = 768, 512
NUM_STEPS        = 30          # P100 is slow; 30 is enough for background fill
GUIDANCE         = 6.0          # low-ish: lean on context, do not invent a subject
N_STEP_SNAPSHOTS = 6            # decoded denoising frames saved for the slides

# ── removal-mask shaping ────────────────────────────────────────────────────
DILATE_PX         = 22          # generous: covers edges/halos of removed people
DILATE_DOWN_EXTRA = 16          # extra downward growth → catches shadow / contact
FEATHER_PX        = 11          # soft composite seam

DETECT_SCORE_THR  = 0.75
MIN_REMOVE_FRAC   = 0.0015      # ignore specks
MAIN_MIN_FRAC     = 0.03        # a "main subject" must occupy at least this area

# ── prompts ─────────────────────────────────────────────────────────────────
PERSON_NEG = ("person, people, man, woman, boy, girl, child, human, tourist, crowd, "
              "face, body, figure, silhouette, pedestrian, ")
BASE_NEG   = "blurry, distorted, low quality, watermark, text, deformed, artifacts, extra limbs"
REMOVAL_NEG = PERSON_NEG + BASE_NEG
BG_PROMPT_TMPL = "empty {bg}, clean continuous background, no people, photorealistic, natural daylight"
NAIVE_PROMPT   = "a photo"      # deliberately generic → shows the re-insertion failure

SEED = 42

# ── per-photo control ───────────────────────────────────────────────────────
# keep_point: (x,y) in full-res px marking the MAIN subject to keep (most reliable).
#             If None, fall back to the most central sufficiently-large person.
# protect_points: points that must NEVER be removed (e.g. fountain statues).
PHOTO_CONFIG = {
    "DSCF5883": {"keep_point": (905, 470),  "protect_points": []},  # Trevi — sunglasses guy center
    "DSCF5909": {"keep_point": (880, 520),  "protect_points": []},  # Navona — white-T guy center-right
    "DSCF5905": {"keep_point": (888, 470),  "protect_points": []},  # Navona — white-T guy center
    "DSCF5873": {"keep_point": (905, 460),  "protect_points": []},  # Trevi — sunglasses guy (alone)
}
print("Globals set.")

## § 3 · Utils

In [ ]:
def set_all_seeds(seed: int) -> None:
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)

def _gen(seed: int):
    g = torch.Generator(device=DEVICE); g.manual_seed(seed); return g

def _font(size=20):
    for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
              "/opt/conda/fonts/DejaVuSans.ttf"]:
        try: return ImageFont.truetype(p, size)
        except Exception: pass
    return ImageFont.load_default()

In [ ]:
# ── Detection: every person via Mask R-CNN ──────────────────────────────────
# detector is loaded in §5. COCO 'person' = class id 1 in torchvision.
PERSON_CLASS = 1

def detect_persons(image: Image.Image, thr: float = DETECT_SCORE_THR) -> list:
    W, H = image.size
    t = TF.to_tensor(image.convert("RGB")).to(DEVICE)
    with torch.no_grad():
        out = detector([t])[0]
    persons = []
    for box, lbl, score, mask in zip(out["boxes"], out["labels"], out["scores"], out["masks"]):
        if int(lbl) != PERSON_CLASS or float(score) < thr:
            continue
        m = (mask[0] > 0.5).cpu().numpy().astype(np.uint8) * 255   # 0/255, not 0/1
        area = int((m > 0).sum())
        if area / float(W * H) < MIN_REMOVE_FRAC:
            continue
        ys, xs = np.where(m > 0)
        persons.append({
            "box": box.cpu().numpy().astype(int), "score": float(score),
            "mask": m, "area": area, "frac": area / float(W * H),
            "cx": float(xs.mean()), "cy": float(ys.mean()),
        })
    persons.sort(key=lambda p: p["area"], reverse=True)
    return persons

def select_main(persons: list, W: int, H: int, cfg: dict):
    if not persons:
        return None
    kp = cfg.get("keep_point")
    if kp is not None:
        # the detected person whose mask contains the keep point, else nearest centroid
        inside = [i for i, p in enumerate(persons) if p["mask"][min(H-1, kp[1]), min(W-1, kp[0])] > 0]
        if inside:
            return max(inside, key=lambda i: persons[i]["area"])
        return min(range(len(persons)),
                   key=lambda i: (persons[i]["cx"] - kp[0])**2 + (persons[i]["cy"] - kp[1])**2)
    # fallback: most central among sufficiently large persons
    big = [i for i, p in enumerate(persons) if p["frac"] >= MAIN_MIN_FRAC] or list(range(len(persons)))
    return min(big, key=lambda i: abs(persons[i]["cx"] / W - 0.5))

def _protected(p, cfg, W, H):
    for (px, py) in cfg.get("protect_points", []):
        if p["mask"][min(H-1, py), min(W-1, px)] > 0:
            return True
    return False

In [ ]:
# ── SAM mask refinement (box-prompted) ──────────────────────────────────────
# sam_model / sam_processor loaded in §5.
def sam_refine_box(image: Image.Image, box) -> np.ndarray:
    img_rgb = np.array(image.convert("RGB"))
    inputs = sam_processor(images=img_rgb,
                           input_boxes=[[[float(box[0]), float(box[1]),
                                          float(box[2]), float(box[3])]]],
                           return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = sam_model(**inputs)
    masks = sam_processor.post_process_masks(out.pred_masks.cpu(),
                                             inputs["original_sizes"].cpu(),
                                             inputs["reshaped_input_sizes"].cpu())
    scores = out.iou_scores[0, 0]
    best = int(scores.argmax())
    return masks[0][0][best].numpy().astype(np.uint8) * 255   # 0/255

def build_removal_mask(image, persons, main_idx, cfg, use_sam=True):
    W, H = image.size
    raw = np.zeros((H, W), np.uint8)
    refined = np.zeros((H, W), np.uint8)
    removed = []
    for i, p in enumerate(persons):
        if i == main_idx or _protected(p, cfg, W, H):
            continue
        raw |= p["mask"]
        if use_sam:
            try:
                refined |= sam_refine_box(image, p["box"])
            except Exception as e:
                print("  [SAM warn]", e); refined |= p["mask"]
        else:
            refined |= p["mask"]
        removed.append(i)
    return raw, refined, removed

def dilate_mask(mask: np.ndarray, px: int = DILATE_PX, down_extra: int = DILATE_DOWN_EXTRA):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*px+1, 2*px+1))
    base = cv2.dilate(mask, k)
    if down_extra > 0:                       # bias the growth downward (shadows)
        shifted = np.zeros_like(base)
        shifted[down_extra:, :] = base[:-down_extra, :]
        base = np.maximum(base, shifted)
    return base

def blackout_persons(image, persons):
    img = np.array(image.convert("RGB")).copy()
    allm = np.zeros(img.shape[:2], np.uint8)
    for p in persons: allm |= p["mask"]
    allm = dilate_mask(allm, 6, 0)
    img[allm > 0] = 127
    return Image.fromarray(img)

In [ ]:
# ── BLIP background captioning ──────────────────────────────────────────────
# blip_model / blip_processor loaded in §5.
_BAD = ("black", "blank", "gray", "grey", "dark")
def caption_background(image_no_people: Image.Image) -> str:
    inputs = blip_processor(images=image_no_people, return_tensors="pt").to(DEVICE, DTYPE)
    with torch.no_grad():
        ids = blip_model.generate(**inputs, max_new_tokens=30)
    cap = blip_processor.decode(ids[0], skip_special_tokens=True).strip()
    if not cap or any(w in cap.lower() for w in _BAD):
        cap = "a street scene with historic architecture"
    return cap

# ── classical pre-fill (background-biased) ──────────────────────────────────
def telea_prefill(image_small: Image.Image, mask_small: Image.Image) -> Image.Image:
    bgr = cv2.cvtColor(np.array(image_small.convert("RGB")), cv2.COLOR_RGB2BGR)
    m = (np.array(mask_small.convert("L")) > 127).astype(np.uint8) * 255
    filled = cv2.inpaint(bgr, m, 5, cv2.INPAINT_TELEA)
    return Image.fromarray(cv2.cvtColor(filled, cv2.COLOR_BGR2RGB))

In [ ]:
# ── Denoising-step capture (for the slides) ─────────────────────────────────
def make_step_callback(store, total, n_snap=N_STEP_SNAPSHOTS):
    want = set(np.linspace(0, total - 1, n_snap).astype(int).tolist())
    def cb(pipe, step, timestep, kw):
        if step in want:
            lat = kw["latents"]
            with torch.no_grad():
                img = pipe.vae.decode(lat / pipe.vae.config.scaling_factor, return_dict=False)[0]
            img = (img / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).float().cpu().numpy()
            store.append((step, Image.fromarray((img * 255).astype(np.uint8))))
        return kw
    return cb

In [ ]:
# ── Visualization helpers ───────────────────────────────────────────────────
def draw_detections(image, persons, main_idx):
    im = image.convert("RGB").copy(); d = ImageDraw.Draw(im); f = _font(28)
    for i, p in enumerate(persons):
        x0, y0, x1, y1 = p["box"]
        keep = (i == main_idx)
        col = (40, 200, 60) if keep else (235, 40, 40)
        d.rectangle([x0, y0, x1, y1], outline=col, width=5)
        tag = f"KEEP #{i}" if keep else f"remove #{i}"
        d.text((x0 + 4, max(0, y0 - 30)), f"{tag} {p['score']:.2f}", fill=col, font=f)
    return im

def mask_overlay(image, mask, color=(235, 40, 40), alpha=0.5):
    im = np.array(image.convert("RGB")).astype(np.float32)
    m = (np.array(Image.fromarray(mask).resize(image.size, Image.NEAREST)) > 127) \
        if mask.shape[:2] != image.size[::-1] else (mask > 127)
    ov = im.copy()
    for c in range(3): ov[..., c] = np.where(m, (1-alpha)*im[..., c] + alpha*color[c], im[..., c])
    return Image.fromarray(ov.astype(np.uint8))

def label_img(img, text, h=34):
    img = img.convert("RGB"); w = img.width
    out = Image.new("RGB", (w, img.height + h), (255, 255, 255))
    out.paste(img, (0, h)); d = ImageDraw.Draw(out); f = _font(22)
    d.text((6, 6), text, fill=(0, 0, 0), font=f); return out

def hstrip(pairs, cell_w=384):
    imgs = [label_img(im.resize((cell_w, int(cell_w*im.height/im.width))), lab) for lab, im in pairs]
    H = max(i.height for i in imgs); W = sum(i.width for i in imgs) + 6*(len(imgs)-1)
    canvas = Image.new("RGB", (W, H), (255, 255, 255)); x = 0
    for i in imgs: canvas.paste(i, (x, 0)); x += i.width + 6
    return canvas

def grid(pairs, cols=3, cell_w=440):
    cells = [label_img(im.resize((cell_w, int(cell_w*im.height/im.width))), lab) for lab, im in pairs]
    rows = math.ceil(len(cells)/cols); cw = cell_w; ch = max(c.height for c in cells)
    canvas = Image.new("RGB", (cols*cw + 8*(cols-1), rows*ch + 8*(rows-1)), (255, 255, 255))
    for k, c in enumerate(cells):
        r, cc = divmod(k, cols); canvas.paste(c, (cc*(cw+8), r*(ch+8)))
    return canvas

def composite_back(original_full, result_small, mask_full):
    res_up = result_small.resize(original_full.size, Image.LANCZOS)
    a = Image.fromarray(mask_full).resize(original_full.size, Image.NEAREST)
    a = a.filter(ImageFilter.GaussianBlur(FEATHER_PX))
    a = np.array(a).astype(np.float32) / 255.0
    o = np.array(original_full.convert("RGB")).astype(np.float32)
    r = np.array(res_up.convert("RGB")).astype(np.float32)
    out = o * (1 - a[..., None]) + r * a[..., None]
    return Image.fromarray(out.clip(0, 255).astype(np.uint8))

## § 4 · Data

In [ ]:
def _find_input_dir() -> Path:
    if INPUT_DIR.exists():
        return INPUT_DIR
    root = Path("/kaggle/input")
    print("INPUT_DIR missing; /kaggle/input contains:",
          [p.name for p in root.iterdir()] if root.exists() else "<none>")
    # any folder under /kaggle/input that holds our DSCF photos
    for p in root.rglob("DSCF*.jpg"):
        return p.parent
    return INPUT_DIR

def load_photos() -> list:
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    src = _find_input_dir()
    samples = []
    for p in sorted(src.iterdir()):
        if p.suffix.lower() not in exts: continue
        img = Image.open(p).convert("RGB")
        samples.append({"image": img, "stem": p.stem, "path": str(p)})
    print(f"Loaded {len(samples)} photos from {src}")
    for s in samples: print(" -", s["stem"], s["image"].size)
    return samples

## § 5 · Network

In [ ]:
# Load all models once. Order: detector → SAM → BLIP → SD-inpaint.
print("Loading Mask R-CNN detector ...")
detector = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT).eval().to(DEVICE)

print("Loading SAM ...")
sam_model = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE).eval()
sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

print("Loading BLIP captioner ...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large", torch_dtype=DTYPE).to(DEVICE).eval()

print("Loading SD-1.5 inpainting ...")
SD_INPAINT_IDS = ["stable-diffusion-v1-5/stable-diffusion-inpainting",
                  "botp/stable-diffusion-v1-5-inpainting",
                  "runwayml/stable-diffusion-inpainting"]
inpaint_pipe = None
for mid in SD_INPAINT_IDS:
    try:
        inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, safety_checker=None).to(DEVICE)
        inpaint_pipe.set_progress_bar_config(disable=True)
        print("  loaded", mid); break
    except Exception as e:
        print("  [warn]", mid, "->", str(e)[:120])
assert inpaint_pipe is not None, "no SD-inpaint checkpoint loaded"
if DEVICE == "cuda":
    print("VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")
print("All models loaded.")

## § 6 · Pipeline
> *No model training in this project.* This section assembles the full automatic
> object-removal pipeline and runs both a **naive** pass (re-inserts a subject) and our
> **removal-tuned** pass (diffuses the background back in), dumping every step for the slides.

In [ ]:
def inpaint(image_small, mask_small, prompt, neg, capture=False, seed=SEED):
    set_all_seeds(seed); g = _gen(seed)
    steps = []
    cb = make_step_callback(steps, NUM_STEPS) if capture else None
    kw = dict(prompt=prompt, negative_prompt=neg, image=image_small, mask_image=mask_small,
              num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE, generator=g,
              width=PROC_W, height=PROC_H)
    if cb is not None:
        kw["callback_on_step_end"] = cb
        kw["callback_on_step_end_tensor_inputs"] = ["latents"]
    res = inpaint_pipe(**kw).images[0]
    return res, steps


def run_removal(sample: dict) -> dict:
    stem = sample["stem"]; orig = sample["image"]; W, H = orig.size
    cfg = PHOTO_CONFIG.get(stem, {})
    d = OUT_DIR / stem; d.mkdir(parents=True, exist_ok=True)
    def save(img, name): img.save(d / name); return img
    t0 = time.perf_counter()

    # 1) detect every person
    persons = detect_persons(orig)
    main_idx = select_main(persons, W, H, cfg)
    print(f"[{stem}] {len(persons)} person(s), main=#{main_idx}")
    save(orig, "00_original.png")
    save(draw_detections(orig, persons, main_idx), "01_detections.png")

    # 2) build removal mask (everyone except main) + SAM refine
    raw, refined, removed = build_removal_mask(orig, persons, main_idx, cfg, use_sam=True)
    if refined.sum() == 0:
        print(f"  nothing to remove for {stem} (clean main-subject shot).")
        save(orig, "11_removal_result.png")
        save(hstrip([("original / kept as-is", orig)]), "99_storyboard.png")
        return {"stem": stem, "removed": 0, "result": orig}

    dil = dilate_mask(refined)
    save(mask_overlay(orig, refined), "03_mask_refined.png")
    save(mask_overlay(orig, dil),     "05_mask_dilated.png")
    masked_in = np.array(orig).copy(); masked_in[dil > 0] = 255
    save(Image.fromarray(masked_in), "06_masked_input.png")

    # 3) background caption (people blacked out)
    no_people = save(blackout_persons(orig, persons), "07_people_blacked.png")
    bg = caption_background(no_people)
    prompt = BG_PROMPT_TMPL.format(bg=bg)
    (d / "08_caption.txt").write_text(f"BLIP background: {bg}\nPrompt: {prompt}\nNeg: {REMOVAL_NEG}")
    print("  caption:", bg)

    # 4) work at diffusion resolution
    img_s  = orig.resize((PROC_W, PROC_H), Image.LANCZOS)
    mask_s = Image.fromarray(dil).resize((PROC_W, PROC_H), Image.NEAREST)
    tight_s = Image.fromarray(refined).resize((PROC_W, PROC_H), Image.NEAREST)

    pre = save(telea_prefill(img_s, mask_s), "09_telea_prefill.png")

    # 5a) NAIVE pass — generic prompt, tight mask, no negative → typically re-inserts a subject
    naive_s, _ = inpaint(img_s, tight_s, NAIVE_PROMPT, "")
    naive_full = composite_back(orig, naive_s, refined)
    save(naive_full, "10_naive_result.png")

    # 5b) TUNED removal pass — bg prompt + person-negative + dilated mask + step capture
    tuned_s, steps = inpaint(img_s, mask_s, prompt, REMOVAL_NEG, capture=True)
    tuned_full = composite_back(orig, tuned_s, dil)
    save(tuned_full, "11_removal_result.png")

    # denoising-step strip
    if steps:
        strip = hstrip([(f"step {s}/{NUM_STEPS}", im) for s, im in steps], cell_w=300)
        save(strip, "12_denoise_steps.png")

    # before / after + storyboard
    save(hstrip([("BEFORE", orig), ("AFTER (removed)", tuned_full)], cell_w=560),
         "13_before_after.png")
    story = grid([
        ("00 original", orig),
        ("01 detect (green=keep)", draw_detections(orig, persons, main_idx)),
        ("05 removal mask (dilated)", mask_overlay(orig, dil)),
        ("09 Telea pre-fill", pre.resize(orig.size)),
        ("10 NAIVE — re-inserts a person", naive_full),
        ("11 OURS — background filled", tuned_full),
    ], cols=3, cell_w=460)
    save(story, "99_storyboard.png")

    print(f"  done in {time.perf_counter()-t0:.1f}s — removed {len(removed)} person(s)")
    if DEVICE == "cuda": torch.cuda.empty_cache(); gc.collect()
    return {"stem": stem, "removed": len(removed), "result": tuned_full,
            "naive": naive_full, "storyboard": story}

## § 7 · Evaluation / Run on the 4 photos

In [ ]:
samples = load_photos()
results = [run_removal(s) for s in samples]

# zip every step image for easy download into the slides
zip_path = "/kaggle/working/removal_steps.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(OUT_DIR.parent))
print("zipped ->", zip_path)
print("steps per photo saved under", OUT_DIR)

In [ ]:
# inline storyboards
for r in results:
    if "storyboard" in r:
        print("="*80, "\n", r["stem"], "— removed", r["removed"], "person(s)")
        plt.figure(figsize=(18, 12)); plt.imshow(r["storyboard"]); plt.axis("off"); plt.show()

In [ ]:
# big before/after for each photo
for r in results:
    if "naive" in r:
        ba = hstrip([("BEFORE", samples[[s['stem'] for s in samples].index(r['stem'])]['image']),
                     ("NAIVE (re-inserts)", r["naive"]),
                     ("OURS (removed)", r["result"])], cell_w=480)
        plt.figure(figsize=(20, 6)); plt.imshow(ba); plt.axis("off")
        plt.title(r["stem"]); plt.show()